# 01. Target Lightcurve Analysis

This notebook demonstrates how to load, visualize, and perform basic analysis on lightcurve data for a specific exoplanet target. We'll explore different photometric bands, identify transit features, and visualize the data to gain initial insights.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --- Configuration ---
TARGET_NAME = "WASP-18"
INSTRUMENT = "muscat"
OBS_DATE = "20230101" # Example date

# --- 1. Load Lightcurve Data ---
# In a real scenario, you would query muscat.db or load from prose2 output files.
# For this demo, we'll create some dummy data.
print(f"Loading lightcurve data for {TARGET_NAME} from {INSTRUMENT} on {OBS_DATE}...")

def generate_dummy_lightcurve(time, period, t0, duration, depth, scatter, band_offset):
    phase = ((time - t0) % period) / period
    # Simple transit model
    flux = np.ones_like(time)
    in_transit = (phase > (0.5 - duration / period / 2)) & (phase < (0.5 + duration / period / 2))
    flux[in_transit] = 1 - depth
    flux += np.random.normal(0, scatter, size=len(time))
    return flux + band_offset

time = np.linspace(0, 1, 500) # Dummy time in days
period = 0.9414529 # WASP-18b period in days
t0 = 0.1 # Arbitrary transit center
duration = 0.009 # Approx 13 minutes
depth = 0.015 # Approx 1.5%
scatter = 0.001

data = {
    'BJD': time,
    'flux_gp': generate_dummy_lightcurve(time, period, t0, duration, depth, scatter, 0.005),
    'flux_rp': generate_dummy_lightcurve(time, period, t0, duration, depth, scatter, 0.000),
    'flux_ip': generate_dummy_lightcurve(time, period, t0, duration, depth, scatter, -0.005),
    'flux_zs': generate_dummy_lightcurve(time, period, t0, duration, depth, scatter, -0.010),
}
df = pd.DataFrame(data)

print("Lightcurve data loaded successfully.")
print(df.head())


## 2. Visualize Lightcurves

We'll plot the raw lightcurves for each band. Note that in a real scenario, these would be differential lightcurves.

In [ ]:
plt.figure(figsize=(12, 8))
for col in df.columns:
    if col.startswith('flux_'):
        band = col.split('_')[1]
        plt.scatter(df['BJD'], df[col], label=f'Band {band}', alpha=0.7, s=5)

plt.xlabel("Barycentric Julian Date (BJD)")
plt.ylabel("Relative Flux")
plt.title(f"Lightcurves for {TARGET_NAME}")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## 3. Basic Binning (Optional)

To reduce noise and highlight features, we can bin the lightcurve data.

In [ ]:
BIN_SIZE_MINUTES = 5 # minutes
bin_size_days = BIN_SIZE_MINUTES / (24 * 60)

binned_dfs = {}
for col in df.columns:
    if col.startswith('flux_'):
        band = col.split('_')[1]
        # Create bins based on BJD
        bins = np.arange(df['BJD'].min(), df['BJD'].max(), bin_size_days)
        labels = bins[:-1] + bin_size_days / 2
        
        df['binned_time'] = pd.cut(df['BJD'], bins=bins, labels=labels, include_lowest=True)
        binned_df = df.groupby('binned_time')[col].mean().reset_index()
        binned_df = binned_df.rename(columns={'binned_time': 'BJD', col: f'binned_{col}'})
        binned_dfs[band] = binned_df

plt.figure(figsize=(12, 8))
for band, binned_df in binned_dfs.items():
    plt.errorbar(binned_df['BJD'], binned_df[f'binned_flux_{band}'], yerr=scatter/np.sqrt(len(time)/len(binned_df)), 
                 fmt='o', capsize=3, label=f'Band {band} Binned', alpha=0.8, markersize=5)

plt.xlabel("Barycentric Julian Date (BJD)")
plt.ylabel("Binned Relative Flux")
plt.title(f"Binned Lightcurves for {TARGET_NAME} ({BIN_SIZE_MINUTES}-min bins)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
